# Jaguar Re-ID — Top-2 Solution: DINOv2 × ConvNeXtV2 late-fusion ensemble

**Author:** jreiml (Kaggle) / zyna (W&B)  
**W&B:** https://wandb.ai/zyna/jaguar-reid-jreiml  
**GitHub:** https://github.com/jreiml/jaguar-re-id

Builds on the top-1 notebook: two independent Phase-2 backbones (DINOv2-ViT-L/14 and ConvNeXtV2-Large) are trained with the same projection head + ArcFace, then their unit-normalized 256-d embeddings are **late-fused by concatenation**. The concatenated 512-d vector is re-normalized, cosine-scored, and re-ranked with the same k-reciprocal post-processing as top-1.

Diversity rationale: DINOv2 is a self-supervised ViT (strong on fine-grained texture via patches); ConvNeXtV2 is a supervised CNN (strong on local hierarchical features). Concatenation avoids needing to calibrate two different score scales.

Expected val mAP (identity-disjoint val_v1): ≈ 0.68 per-model → 0.70+ after concat + rerank.

In [ ]:
!pip install --quiet --no-deps timm==1.0.26 imagehash==4.3.2
from pathlib import Path
import math, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from PIL import Image
import timm
from tqdm.auto import tqdm

DATA_DIR = Path('/kaggle/input/round-2-jaguar-reidentification-challenge')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
WORK = Path('/kaggle/working')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BACKBONES = [
    ('vit_large_patch14_reg4_dinov2.lvd142m', 518),
    ('convnextv2_large.fcmae_ft_in22k_in1k_384', 384),
]

In [ ]:
def load_rgb(path, mode='as_is'):
    img = Image.open(path)
    if img.mode != 'RGBA': return img.convert('RGB')
    arr = np.asarray(img); rgb = arr[..., :3].copy(); alpha = arr[..., 3]
    if mode == 'as_is': return Image.fromarray(rgb)
    rgb[alpha == 0] = {'gray': 128, 'imagenet_mean': None}[mode] if mode != 'imagenet_mean' else np.array([124,116,104], dtype=np.uint8)
    return Image.fromarray(rgb)

class ImageDataset(Dataset):
    def __init__(self, paths, size, mode='as_is'):
        self.paths = paths; self.mode = mode
        self.pp = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.pp(load_rgb(self.paths[i], self.mode)), self.paths[i].name

In [ ]:
# Build identity-disjoint split (shared across both backbones)
df = pd.read_csv(DATA_DIR / 'train.csv')
counts = df.groupby('ground_truth').size()
eligible = counts[counts >= 2].index.tolist()
rng = random.Random(SEED)
cand = sorted(set(eligible)); rng.shuffle(cand)
val_ids = sorted(cand[:max(1, int(round(len(cand) * 0.2)))])
train_ids = sorted(set(eligible) - set(val_ids))
train_fns = df[df.ground_truth.isin(train_ids)].filename.astype(str).tolist()
val_fns = df[df.ground_truth.isin(val_ids)].filename.astype(str).tolist()
by_fn = dict(zip(df.filename.astype(str), df.ground_truth.astype(str)))
id2cls = {i: c for c, i in enumerate(sorted(train_ids))}
train_lab = np.array([id2cls[by_fn[f]] for f in train_fns])
val_labels = np.array([by_fn[f] for f in val_fns])
print(f'train: {len(train_fns)} / {len(train_ids)} ids   val: {len(val_fns)} / {len(val_ids)} ids')

In [ ]:
# Head modules
class Projection(nn.Module):
    def __init__(self, in_d, h, out_d, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(True), nn.Dropout(drop),
            nn.Linear(h, out_d), nn.BatchNorm1d(out_d))
    def forward(self, x): return self.net(x)

class ArcFace(nn.Module):
    def __init__(self, d, nc, m=0.5, s=64.0):
        super().__init__(); self.w = nn.Parameter(torch.empty(nc, d)); nn.init.xavier_uniform_(self.w)
        self.m = m; self.s = s
        self.cm = math.cos(m); self.sm = math.sin(m); self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, e, y):
        e = F.normalize(e, 2, 1); w = F.normalize(self.w, 2, 1)
        c = F.linear(e, w).clamp(-1, 1); sn = torch.sqrt(torch.clamp(1 - c**2, min=1e-9))
        phi = torch.where(c > self.th, c * self.cm - sn * self.sm, c - self.mm)
        oh = torch.zeros_like(c); oh.scatter_(1, y.view(-1,1).long(), 1)
        return ((oh * phi) + ((1-oh) * c)) * self.s

def train_one_backbone(name, size, epochs=25, bs_feat=64, bs_img=16):
    bb = timm.create_model(name, pretrained=True, num_classes=0).eval().to(device)
    with torch.no_grad():
        fd = int(bb(torch.zeros(1, 3, size, size, device=device)).shape[1])
    @torch.no_grad()
    def extract(paths, mode='as_is'):
        dl = DataLoader(ImageDataset(paths, size, mode), batch_size=bs_img, num_workers=2, pin_memory=True)
        return np.vstack([bb(x.to(device)).cpu().numpy() for x, _ in tqdm(dl, desc=f'{name[:16]}-{mode}')])
    tr = extract([TRAIN_DIR / f for f in train_fns])
    va = extract([TRAIN_DIR / f for f in val_fns])
    # Head training
    proj = Projection(fd, 512, 256).to(device); head = ArcFace(256, len(id2cls)).to(device)
    opt = torch.optim.AdamW(list(proj.parameters()) + list(head.parameters()), lr=1e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
    ce = nn.CrossEntropyLoss()
    ds = TensorDataset(torch.from_numpy(tr).float(), torch.from_numpy(train_lab).long())
    dl = DataLoader(ds, batch_size=bs_feat, shuffle=True)
    vt = torch.from_numpy(va).float().to(device)
    best_map = -1; best_state = None; patience = 0
    for ep in range(1, epochs + 1):
        proj.train(); head.train()
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            loss = ce(head(proj(x), y), y)
            opt.zero_grad(); loss.backward(); opt.step()
        proj.eval()
        with torch.no_grad():
            v = F.normalize(proj(vt), 2, 1).cpu().numpy()
        # identity-balanced mAP
        sim = v @ v.T; np.fill_diagonal(sim, -np.inf)
        per = {}
        for i in range(len(val_labels)):
            m = (val_labels == val_labels[i]).astype(int); m[i] = 0
            if m.sum() == 0: continue
            o = np.argsort(-sim[i]); sm = m[o]; prec = np.cumsum(sm) / np.arange(1, len(sm)+1)
            per.setdefault(val_labels[i], []).append(float((prec*sm).sum()/sm.sum()))
        vm = float(np.mean([np.mean(v_) for v_ in per.values()]))
        sch.step(vm)
        if vm > best_map: best_map = vm; best_state = {'proj': proj.state_dict()}; patience = 0
        else: patience += 1
        print(f'  {name[:16]} ep {ep:03d} val_mAP {vm:.4f} best {best_map:.4f}')
        if patience >= 8: break
    proj.load_state_dict(best_state['proj'])
    return bb, proj, size

models = []
for name, size in BACKBONES:
    bb, proj, sz = train_one_backbone(name, size)
    models.append((bb, proj, sz, name))

In [ ]:
# Concatenate normalized embeddings over backbones, then rerank + score pairs
pairs = pd.read_csv(DATA_DIR / 'test.csv')
unique = sorted(set(pairs.query_image).union(set(pairs.gallery_image)))

all_embs = []
for bb, proj, size, name in models:
    @torch.no_grad()
    def extract(paths):
        dl = DataLoader(ImageDataset(paths, size, 'gray'), batch_size=16, num_workers=2, pin_memory=True)
        raw = np.vstack([bb(x.to(device)).cpu().numpy() for x, _ in tqdm(dl, desc=f'{name[:16]}-test')])
        raw = torch.from_numpy(raw).float().to(device)
        return F.normalize(proj(raw), 2, 1).cpu().numpy()
    all_embs.append(extract([TEST_DIR / f for f in unique]))

emb = np.concatenate(all_embs, axis=1)
emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
idx = {fn: i for i, fn in enumerate(unique)}

# Same k-reciprocal rerank as top-1 (k1=35, k2=6, λ=0.2) — conservative mid-range
def rerank(emb, k1=35, k2=6, lam=0.2):
    od = np.maximum(1 - emb @ emb.T, 0.0); od = od / np.maximum(od.max(axis=0, keepdims=True), 1e-12)
    ir = np.argsort(od, axis=1, kind='stable'); n = len(emb); V = np.zeros_like(od, dtype=np.float32)
    def kr(i, k):
        fw = ir[i, :k+1]; bw = ir[fw, :k+1]; return fw[np.where(bw == i)[0]]
    for i in range(n):
        kri = kr(i, k1); ke = kri.copy()
        for c in kri:
            s = kr(c, int(round(k1/2)))
            if len(np.intersect1d(s, kri)) > 2.0/3.0 * len(s): ke = np.union1d(ke, s)
        w = np.exp(-od[i, ke]).astype(np.float32); V[i, ke] = w / w.sum()
    if k2 > 1:
        Vq = np.zeros_like(V)
        for i in range(n): Vq[i] = np.mean(V[ir[i, :k2], :], axis=0)
        V = Vq
    inv = [np.where(V[:, c] != 0)[0] for c in range(n)]
    jd = np.zeros_like(od)
    for i in range(n):
        tm = np.zeros(n, dtype=np.float32); nz = np.where(V[i] != 0)[0]
        for j_, c in enumerate(nz):
            idx_set = inv[c]; tm[idx_set] += np.minimum(V[i, c], V[idx_set, c])
        jd[i] = 1.0 - tm / (2.0 - tm)
    fd = jd * (1-lam) + od * lam; np.fill_diagonal(fd, 0.0); return fd

dist = rerank(emb)
sim_mat = 1.0 - (dist - dist.min()) / max(dist.max() - dist.min(), 1e-12)
sim = np.array([sim_mat[idx[q], idx[g]] for q, g in zip(pairs.query_image, pairs.gallery_image)])
sim = np.clip(sim, 0, 1)
pd.DataFrame({'row_id': pairs.row_id.values, 'similarity': sim}).to_csv(WORK / 'submission.csv', index=False)
print('Saved', WORK / 'submission.csv', 'rows:', len(sim))